In [1]:
import pandas as pd
import numpy as np

In [12]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_103_CRRI_Mathura_Road_Delhi_IMD_1Day.csv")

In [13]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,133.07,202.10,37.56,19.00,56.56,NaN,NaN,0.72,33.54,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2024-01-02,162.02,251.05,38.69,19.23,57.92,NaN,NaN,0.75,33.71,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2024-01-03,148.37,217.99,41.03,19.72,60.75,NaN,NaN,0.75,33.51,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2024-01-04,183.98,274.92,36.60,18.75,53.82,NaN,NaN,0.75,34.28,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2024-01-05,130.75,201.58,39.39,19.27,58.66,NaN,NaN,0.40,34.45,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,174.15,224.80,33.17,33.66,41.60,NaN,NaN,1.76,26.51,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
362,2024-12-28,79.93,130.71,37.09,25.91,41.24,NaN,NaN,0.90,8.15,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
363,2024-12-29,85.71,151.43,58.93,26.83,62.09,NaN,NaN,0.51,17.19,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
364,2024-12-30,87.82,161.22,55.06,30.16,60.80,NaN,NaN,0.62,19.69,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [14]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 11)


In [15]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['NH3 (µg/m³)', 'Xylene (µg/m³)']
Dropped rows (>70% NaN): 69
Missing values after imputation:
 Timestamp        0
PM2.5 (µg/m³)    0
PM10 (µg/m³)     0
NO (µg/m³)       0
NO2 (µg/m³)      0
NOx (ppb)        0
CO (mg/m³)       0
Ozone (µg/m³)    0
TOT-RF (mm)      0
dtype: int64


In [16]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [17]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (297, 9)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         133.07        202.10       37.56        19.00   
1  2024-01-02         162.02        251.05       38.69        19.23   
2  2024-01-03         148.37        217.99       41.03        19.72   
3  2024-01-04         183.98        274.92       36.60        18.75   
4  2024-01-05         130.75        201.58       39.39        19.27   

   NOx (ppb)  CO (mg/m³)  Ozone (µg/m³)  TOT-RF (mm)  
0      56.56        0.72          33.54          0.0  
1      57.92        0.75          33.71          0.0  
2      60.75        0.75          33.51          0.0  
3      53.82        0.75          34.28          0.0  
4      58.66        0.40          34.45          0.0  


In [18]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [19]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),CO (mg/m³),Ozone (µg/m³),TOT-RF (mm)
0,2024-01-01,0.739072,0.063941,0.409233,0.142987,0.562585,-0.727194,0.686669,0.0
1,2024-01-02,1.257942,0.624588,0.458034,0.165756,0.614154,-0.680147,0.701025,0.0
2,2024-01-03,1.013294,0.245937,0.559091,0.214265,0.721462,-0.680147,0.684135,0.0
3,2024-01-04,1.651530,0.897983,0.367774,0.118237,0.458689,-0.680147,0.749162,0.0
4,2024-01-05,0.697491,0.057985,0.488265,0.169716,0.642213,-1.229028,0.763519,0.0
...,...,...,...,...,...,...,...,...,...
292,2024-12-27,1.475348,0.323935,0.219645,1.594294,-0.004671,0.903766,0.092976,0.0
293,2024-12-28,-0.213354,-0.753723,0.388936,0.827061,-0.018322,-0.444913,-1.457549,0.0
294,2024-12-29,-0.109759,-0.516407,1.332129,0.918139,0.772273,-1.056523,-0.694110,0.0
295,2024-12-30,-0.071942,-0.404277,1.164997,1.247802,0.723358,-0.884017,-0.482982,0.0


In [20]:
df.to_excel('northcampus2024.xlsx', index=False)